# Titanic EDA
Exploratory Data Analysis of the Titanic dataset.
Goal: Understand the data before cleaning and feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('Titanic-Dataset.csv')
print('Shape:', df.shape)

## 1. Data Overview

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
print('Duplicate rows:', df.duplicated().sum())

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).query('`Missing Count` > 0')

In [ ]:
missing_pct.drop(missing_pct[missing_pct == 0].index).sort_values(ascending=False).plot(
    kind='bar', color='coral', title='Missing Value % per Column'
)
plt.ylabel('% Missing')
plt.tight_layout()
plt.show()

In [ ]:
print('Age missing by Pclass:')
print(df[df['Age'].isnull()]['Pclass'].value_counts())
print('\nCabin missing by Pclass:')
print(df[df['Cabin'].isnull()]['Pclass'].value_counts())

## 3. Target Variable — Survived

In [ ]:
print(df['Survived'].value_counts())
print('\nSurvival Rate:', df['Survived'].mean().round(4) * 100, '%')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['Survived'].value_counts().plot(kind='bar', ax=axes[0], color=['salmon','steelblue'],
                                   title='Survived Count')
axes[0].set_xticklabels(['Not Survived (0)', 'Survived (1)'], rotation=0)
df['Survived'].value_counts().plot(kind='pie', ax=axes[1], labels=['Not Survived','Survived'],
                                   autopct='%1.1f%%', colors=['salmon','steelblue'],
                                   title='Survival Proportion')
plt.tight_layout()
plt.show()

## 4. Categorical Features vs Survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Sex', hue='Survived', data=df, ax=axes[0])
axes[0].set_title('Survival Count by Sex')
df.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1], color=['steelblue','salmon'])
axes[1].set_title('Survival Rate by Sex')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['Female', 'Male'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Pclass', hue='Survived', data=df, ax=axes[0])
axes[0].set_title('Survival Count by Pclass')
df.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[1], color=['gold','silver','sienna'])
axes[1].set_title('Survival Rate by Pclass')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Embarked', hue='Survived', data=df, ax=axes[0])
axes[0].set_title('Survival Count by Embarked Port')
df.groupby('Embarked')['Survived'].mean().plot(kind='bar', ax=axes[1])
axes[1].set_title('Survival Rate by Embarked Port')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['C (Cherbourg)', 'Q (Queenstown)', 'S (Southampton)'], rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.groupby(['Sex', 'Pclass'])['Survived'].mean().unstack()
print('Survival Rate by Sex + Pclass:\n', pivot)

sns.catplot(x='Pclass', hue='Survived', col='Sex', data=df, kind='count', height=5)
plt.suptitle('Survival by Sex and Pclass', y=1.02)
plt.show()

## 5. Numerical Features Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.histplot(df['Age'].dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title('Age Distribution')

sns.histplot(data=df.dropna(subset=['Age']), x='Age', hue='Survived', bins=30, kde=True, ax=axes[1])
axes[1].set_title('Age Distribution by Survival')

sns.boxplot(x='Survived', y='Age', data=df, ax=axes[2])
axes[2].set_title('Age Boxplot by Survival')
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(x='Pclass', y='Age', data=df)
plt.title('Age Distribution by Pclass')
plt.show()
print('Median Age by Pclass:')
print(df.groupby('Pclass')['Age'].median())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.histplot(df['Fare'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('Fare Distribution (Raw)')

sns.histplot(np.log1p(df['Fare']), bins=40, kde=True, ax=axes[1])
axes[1].set_title('Fare Distribution (Log Scale)')

sns.boxplot(x='Survived', y='Fare', data=df, ax=axes[2])
axes[2].set_title('Fare by Survival')
plt.tight_layout()
plt.show()

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.countplot(x='SibSp', hue='Survived', data=df, ax=axes[0])
axes[0].set_title('SibSp vs Survival')

sns.countplot(x='Parch', hue='Survived', data=df, ax=axes[1])
axes[1].set_title('Parch vs Survival')

family_surv = df.groupby('FamilySize')['Survived'].mean()
family_surv.plot(kind='bar', ax=axes[2], color='teal')
axes[2].set_title('Survival Rate by Family Size')
axes[2].set_ylabel('Survival Rate')
axes[2].set_xlabel('Family Size')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
num_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']
corr = df[num_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Advanced EDA — Title Extraction & Age Binning

In [ ]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
print('Title Value Counts:\n', df['Title'].value_counts())

title_surv = df.groupby('Title')['Survived'].mean().sort_values(ascending=False)
title_surv.plot(kind='bar', figsize=(12, 5), color='mediumpurple',
                title='Survival Rate by Title')
plt.ylabel('Survival Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df['AgeBin'] = pd.cut(df['Age'],
                      bins=[0, 12, 18, 35, 60, 100],
                      labels=['Child', 'Teen', 'Adult', 'Middle-Aged', 'Senior'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='AgeBin', hue='Survived', data=df, ax=axes[0])
axes[0].set_title('Survival Count by Age Group')

df.groupby('AgeBin', observed=True)['Survived'].mean().plot(
    kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Survival Rate by Age Group')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 8. Pairplot — Big Picture View

In [ ]:
pair_cols = ['Survived', 'Age', 'Fare', 'Pclass', 'FamilySize']
sns.pairplot(df[pair_cols].dropna(), hue='Survived', diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot of Key Numerical Features', y=1.02)
plt.show()

## 9. EDA Summary — Key Findings

| Feature | Finding |
|---|---|
| **Sex** | Females survived at ~74% vs males at ~19% |
| **Pclass** | 1st class: ~63% survived, 3rd class: ~24% |
| **Age** | Children had higher survival; age partially missing |
| **Fare** | Higher fare → higher survival (correlated with Pclass) |
| **FamilySize** | Solo travelers and large families (>4) fared worse |
| **Embarked** | Cherbourg (C) had highest survival rate |
| **Title** | Miss/Mrs/Master had much higher survival than Mr |
| **Missing** | Age: 20% missing (fill by Pclass median), Cabin: 77% missing (create binary flag) |